In [ ]:
#imports and config
modules = ['torch.', 'torchvision', 'sklearn', 'matplotlib.', 'seaborn','numpy','pandas']

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

device = torch.device ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")


LATENT_DIM=64
BATCH_SIZE=64
LR=0.0002
EPOCHS=30
IMAGE_SIZE=28

In [ ]:
#Load MNIST Data
transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
])

dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)

loader = DataLoader(dataset, batch_size=BATCH_SIZE,shuffle=True)
print(f"Using: {device}")

In [ ]:
#generator
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(LATENT_DIM, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, IMAGE_SIZE * IMAGE_SIZE),
            nn.Tanh()
        )

    def forward(self, z):
        return self.model(z).view(-1, 1, IMAGE_SIZE, IMAGE_SIZE)

In [ ]:
#discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(IMAGE_SIZE * IMAGE_SIZE, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
#test
G = Generator().to(device)
D = Discriminator().to(device)

z= torch.randn(1,LATENT_DIM).to(device)
out = G(z)
print("Generator output shape:",out.shape)

In [ ]:
#training loop
G = Generator().to(device)
D = Discriminator().to(device)

opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
criterion = nn.BCELoss()

for epoch in range(EPOCHS):
    for real_imgs, _ in loader:
        real_imgs = real_imgs.to(device)
        batch = real_imgs.size(0)

        # --- Train Discriminator ---
        real_labels = torch.ones(batch, 1).to(device)
        fake_labels = torch.zeros(batch, 1).to(device)

        z = torch.randn(batch, LATENT_DIM).to(device)
        fake_imgs = G(z)

        loss_D = criterion(D(real_imgs), real_labels) + \
                 criterion(D(fake_imgs.detach()), fake_labels)
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()

        # --- Train Generator ---
        loss_G = criterion(D(fake_imgs), real_labels)
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()

    print(f"Epoch {epoch+1}/{EPOCHS} | D Loss: {loss_D.item():.4f} | G Loss: {loss_G.item():.4f}")

In [ ]:
#view generated images
G.eval()
with torch.no_grad():
    z = torch.randn(16, LATENT_DIM).to(device)
    samples = G(z).cpu().squeeze()

fig, axes = plt.subplots(4,4, figsize=(6,6))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i], cmap="gray", vmin=-1, vmax=1)
    ax.axis("off")
plt.tight_layout()
plt.show()